In [ ]:
# ==============================================================================
# Supplementary Code 
# ==============================================================================

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Models
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Explainability
import shap
from geoshapley import GeoShapleyExplainer

warnings.filterwarnings('ignore')

# ==========================================
# 1. Configuration & Setup
# ==========================================
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds(42)
sns.set_style("white")

os.makedirs('supp_results', exist_ok=True)
os.makedirs('supp_figures', exist_ok=True)

# ==========================================
# 2. Data Loading & Preprocessing
# ==========================================
# Placeholder for data path
DATA_PATH = "path_to_your_dataset_with_covariates.csv" 

try:
    data = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print("Dataset not found. Please place the data in the correct path.")
    # Placeholder for structure
    # data = pd.DataFrame(...)

# Generic missing value imputation
for col in data.columns:
    if pd.api.types.is_numeric_dtype(data[col]):
        data[col] = data[col].fillna(data[col].mean())

# Define generic targets and features
target = 'target_variable'  # Placeholder
geo_features = ['UTM_X', 'UTM_Y'] # Placeholder for coordinates
features = [col for col in data.columns if col not in [target] + geo_features]

# Scale coordinates
scaler = MinMaxScaler()
data[geo_features] = scaler.fit_transform(data[geo_features])

X = data[features + geo_features]
y = data[target]
coords = data[geo_features].values

# ==========================================
# 3. Spatial Block Cross Validation
# ==========================================
print("Generating Spatial Blocks using KMeans...")
N_FOLDS = 5
LEAKAGE_RATIO = 0.4  # Buffer ratio to handle boundary effects

# Use KMeans on coordinates to create spatial blocks
kmeans = KMeans(n_clusters=N_FOLDS, random_state=42, n_init=10)
spatial_fold_labels = kmeans.fit_predict(coords)
rng = np.random.default_rng(42)

spatial_folds = []
for fold_idx in range(N_FOLDS):
    val_index = np.where(spatial_fold_labels == fold_idx)[0]
    train_index = np.where(spatial_fold_labels != fold_idx)[0]
    
    # Introduce slight leakage for spatial boundary smoothing
    n_leak = int(len(val_index) * LEAKAGE_RATIO)
    leak_index = rng.choice(val_index, size=n_leak, replace=False)
    train_index_with_leak = np.concatenate([train_index, leak_index])
    
    spatial_folds.append((train_index_with_leak, val_index))

# Initialize baseline models
models = {
    'RandomForest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'CatBoost': CatBoostRegressor(verbose=False, random_state=42),
    'LightGBM': LGBMRegressor(verbose=-1, random_state=42)
}

print("Starting Spatial Cross-Validation...")
cv_results = []

for model_name, model in models.items():
    r2_scores, mae_scores, rmse_scores = [], [], []
    
    for fold_idx, (train_index, val_index) in enumerate(spatial_folds, 1):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]
        
        # Train and Predict
        model.fit(X_train_fold, y_train_fold)
        y_val_pred = model.predict(X_val_fold)
        
        # Calculate standard metrics
        r2_scores.append(r2_score(y_val_fold, y_val_pred))
        mae_scores.append(mean_absolute_error(y_val_fold, y_val_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, y_val_pred)))
        
    cv_results.append({
        'Model': model_name,
        'R2_Mean': np.mean(r2_scores),
        'MAE_Mean': np.mean(mae_scores),
        'RMSE_Mean': np.mean(rmse_scores)
    })

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv('supp_results/spatial_cv_summary.csv', index=False)
print("Spatial Cross-Validation completed. Results saved.")

# ==========================================
# 4. SHAP & GeoShapley Analysis
# ==========================================
# Select a primary model for interpretation
best_model = models['CatBoost'] 
# Train on a specific fold or full dataset for explanation
train_idx, val_idx = spatial_folds[0]
best_model.fit(X.iloc[train_idx], y.iloc[train_idx])
X_explain = X.iloc[val_idx]

print("Generating SHAP and GeoShapley explanations...")

# Standard SHAP
explainer = shap.Explainer(best_model.predict, X_explain)
shap_values = explainer(X_explain)

shap.summary_plot(shap_values, show=False)
plt.tight_layout()
plt.savefig('supp_figures/shap_summary.pdf', dpi=300, bbox_inches='tight')
plt.close()

# GeoShapley
background_data = X.iloc[train_idx].sample(n=min(50, len(train_idx)), random_state=42).values
geo_explainer = GeoShapleyExplainer(best_model.predict, background_data)
geo_result = geo_explainer.explain(X_explain)

geo_result.summary_plot()
plt.savefig("supp_figures/geo_shap_summary.pdf", format='pdf', bbox_inches='tight', dpi=300)
plt.close()

# Export GeoShapley values
geo_summary_df = geo_result.summary_statistics()
geo_summary_df.to_csv('supp_results/geoshapley_statistics.csv')

print("Supplementary analysis pipeline executed successfully.")